Replicated the Dataset

In [2]:
import shutil
from pathlib import Path

def replicate_dataset(source_name="Sample_Training", target_name="Sample_Training_Balanced"):
    """
    Safely clones the entire dataset directory to a new location.
    If the target already exists, it warns you instead of crashing.
    """
    source_dir = Path(source_name)
    target_dir = Path(target_name)

    print(f"=== Starting Dataset Replication ===")
    
    # 1. Check if source exists
    if not source_dir.exists():
        raise FileNotFoundError(f"CRITICAL ERROR: Could not find '{source_name}'. Are you in the right directory?")

    # 2. Check if target already exists (Safety First!)
    if target_dir.exists():
        print(f"Warning: '{target_name}' already exists.")
        print("To prevent accidental overwrites, please delete it manually if you want a fresh start.")
        return target_dir

    # 3. Perform the exact clone
    print(f"Cloning '{source_name}' to '{target_name}'...")
    print("This might take a few seconds depending on the number of CSVs...")
    
    shutil.copytree(source_dir, target_dir)
    
    # 4. Verify the clone
    original_files = list(source_dir.rglob("*.csv"))
    cloned_files = list(target_dir.rglob("*.csv"))
    
    print("\n=== Replication Successful! ===")
    print(f"Original CSVs: {len(original_files)}")
    print(f"Cloned CSVs:   {len(cloned_files)}")
    print(f"Your safe workspace is ready at: ./{target_name}/")
    
    return target_dir

# Execute the replication
balanced_workspace = replicate_dataset("./DataSet/Sample_Training", "./Balanced_DataSets/Sample_Training_Balanced")

=== Starting Dataset Replication ===
Cloning './DataSet/Sample_Training' to './Balanced_DataSets/Sample_Training_Balanced'...
This might take a few seconds depending on the number of CSVs...

=== Replication Successful! ===
Original CSVs: 2473
Cloned CSVs:   2473
Your safe workspace is ready at: ././Balanced_DataSets/Sample_Training_Balanced/


handling imbalance using Magnitude scaling and using jitter

### Time-Series Data Augmentation for Fall Detection

#### Why Not SMOTE?

SMOTE destroys temporal physics — it doesn't understand sequential sensor data.

#### Our Approach: Two Mathematical Transformations

---

#### 1. Magnitude Scaling (Force Variance)

**The Physics:** Different people fall with different force (heavy vs. light individuals)

**The Math:**
$$\mathbf{X}_{scaled} = \alpha \cdot \mathbf{X}, \quad \alpha \in [0.8, 1.2]$$

**What it teaches:** Falls are defined by **shape**, not absolute numeric thresholds

---

#### 2. Jittering (Sensor Noise Injection)

**The Physics:** Real sensors have micro-vibrations and electrical noise

**The Math:**
$$\mathbf{X}_{jittered} = \mathbf{X} + \mathbf{N}, \quad \mathbf{N} \sim \mathcal{N}(0, \sigma^2)$$

**What it does:** Prevents memorization of irrelevant micro-fluctuations (acts like Dropout)

---

#### Results

| Technique | Effect |
|-----------|--------|
| Magnitude Scaling | Stretches/compresses peaks |
| Jittering | Adds controlled randomness |
| Combined | **Triples minority class size** |

#### Key Insight

> The network learns the **generalized physical pattern** of a fall — not memorized numeric values from the training set.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

print("=== Starting Physics-Based Data Augmentation ===")

def augment_dataset(balanced_dir_name="Sample_Training_Balanced"):
    target_dir = Path(balanced_dir_name)
    
    if not target_dir.exists():
        raise FileNotFoundError(f"CRITICAL ERROR: Folder '{balanced_dir_name}' not found! Did you run the replication script?")
        
    all_files = list(target_dir.rglob("*.csv"))
    print(f"Found {len(all_files)} total CSVs in the workspace.")
    
    # Define which columns represent physical sensors (we don't augment time or labels)
    sensor_cols = ['AccX', 'AccY', 'AccZ', 'GyrX', 'GyrY', 'GyrZ', 'EulerX', 'EulerY', 'EulerZ']
    
    falls_augmented = 0
    
    # Wrap in tqdm for a nice progress bar
    for file_path in tqdm(all_files, desc="Scanning and Augmenting"):
        df = pd.read_csv(file_path)
        
        # Check if this specific CSV contains an actual fall
        if df['FallCheck'].sum() > 0:
            
            # ---------------------------------------------------
            # AUGMENTATION 1: Scale Up (Simulates heavier impact)
            # ---------------------------------------------------
            df_heavy = df.copy()
            scale_factor_up = np.random.uniform(1.05, 1.15) # 5% to 15% increase
            df_heavy[sensor_cols] = df_heavy[sensor_cols] * scale_factor_up
            
            # Save the new file
            new_path_heavy = file_path.parent / f"{file_path.stem}_AUG_Heavy.csv"
            df_heavy.to_csv(new_path_heavy, index=False)
            
            # ---------------------------------------------------
            # AUGMENTATION 2: Scale Down (Simulates lighter impact)
            # ---------------------------------------------------
            df_light = df.copy()
            scale_factor_down = np.random.uniform(0.85, 0.95) # 5% to 15% decrease
            df_light[sensor_cols] = df_light[sensor_cols] * scale_factor_down
            
            new_path_light = file_path.parent / f"{file_path.stem}_AUG_Light.csv"
            df_light.to_csv(new_path_light, index=False)
            
            # ---------------------------------------------------
            # AUGMENTATION 3: Jitter (Simulates cheap hardware noise)
            # ---------------------------------------------------
            df_noisy = df.copy()
            # Generate 2% noise based on the standard deviation of each column
            noise = np.random.normal(0, 0.02, size=df_noisy[sensor_cols].shape)
            df_noisy[sensor_cols] = df_noisy[sensor_cols] + noise
            
            new_path_noisy = file_path.parent / f"{file_path.stem}_AUG_Noisy.csv"
            df_noisy.to_csv(new_path_noisy, index=False)
            
            falls_augmented += 1
            
    print("\n=== Augmentation Complete! ===")
    print(f"Original files containing falls: {falls_augmented}")
    print(f"Total NEW augmented files generated: {falls_augmented * 3}")
    
    # Count the new total
    final_files = list(target_dir.rglob("*.csv"))
    print(f"New total CSV count in '{balanced_dir_name}': {len(final_files)}")

# Run the engine
augment_dataset("./Balanced_DataSets/Sample_Training_Balanced")

=== Starting Physics-Based Data Augmentation ===
Found 2473 total CSVs in the workspace.


Scanning and Augmenting: 100%|██████████| 2473/2473 [00:50<00:00, 48.89it/s] 


=== Augmentation Complete! ===
Original files containing falls: 1862
Total NEW augmented files generated: 5586
New total CSV count in './Balanced_DataSets/Sample_Training_Balanced': 8059


In [4]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

print("=== Starting Exploratory Data Analysis (EDA) ===")

def analyze_dataset(dataset_path="Sample_Training_Balanced"):
    target_dir = Path(dataset_path)
    
    if not target_dir.exists():
        raise FileNotFoundError(f"CRITICAL ERROR: Folder '{dataset_path}' not found!")
        
    all_files = list(target_dir.rglob("*.csv"))
    print(f"Total CSV files found: {len(all_files)}\n")
    
    # ---------------------------------------------------
    # Step 1: Describe the Data (Statistical Summary)
    # ---------------------------------------------------
    # We will grab the first augmented file we find to show the stats
    sample_file = None
    for file in all_files:
        if "AUG" in file.name:
            sample_file = file
            break
            
    if sample_file:
        print(f"--- Step 1: Statistical Summary of an Augmented File ---")
        print(f"File: {sample_file.name}")
        df_sample = pd.read_csv(sample_file)
        # Displaying the describe() table, rounding to 3 decimal places for readability
        print(df_sample.describe().round(3).to_string())
        print("\n" + "="*50 + "\n")
    
    # ---------------------------------------------------
    # Step 2: Count 1s and 0s Across the WHOLE Dataset
    # ---------------------------------------------------
    print("--- Step 2: Global Class Distribution ---")
    
    total_zeros = 0
    total_ones = 0
    
    for file in tqdm(all_files, desc="Counting Labels"):
        df = pd.read_csv(file)
        
        # Count the occurrences of 1 and 0 in the 'FallCheck' column
        counts = df['FallCheck'].value_counts()
        
        # .get() is safer in case a file is 100% normal or 100% fall
        total_zeros += counts.get(0, 0)
        total_ones += counts.get(1, 0)
        
    total_frames = total_zeros + total_ones
    
    print("\n--- Final Dataset Balance ---")
    print(f"Total Normal Frames (0s): {total_zeros:,}")
    print(f"Total Fall Frames (1s):   {total_ones:,}")
    print(f"Total Combined Frames:    {total_frames:,}")
    
    # Calculate the new ratio
    if total_ones > 0:
        ratio = total_zeros / total_ones
        print(f"\nNew Frame-Level Ratio (Normal : Fall) -> {ratio:.2f} : 1")
    else:
        print("\nWarning: No falls (1s) found in the dataset!")

# Run the analysis
analyze_dataset("./Balanced_DataSets/Sample_Training_Balanced")

=== Starting Exploratory Data Analysis (EDA) ===
Total CSV files found: 8059

--- Step 1: Statistical Summary of an Augmented File ---
File: S11T28R03_AUG_Heavy.csv
       TimeStamp(s)  FrameCounter     AccX     AccY     AccZ     GyrX     GyrY     GyrZ   EulerX   EulerY   EulerZ  FallCheck
count       474.000       474.000  474.000  474.000  474.000  474.000  474.000  474.000  474.000  474.000  474.000    474.000
mean          2.375       237.500   -0.006   -1.012   -0.045  -21.782    3.407   -3.302   90.590   -0.340  -17.901      0.169
std           1.370       136.976    0.295    0.495    0.427   72.169   35.046   16.038   20.551    3.654    5.254      0.375
min           0.010         1.000   -1.191   -3.100   -4.292 -350.572 -134.913  -63.641   -6.365   -4.300  -27.690      0.000
25%           1.192       119.250   -0.077   -1.111   -0.047  -11.885   -8.572   -7.653   92.781   -1.982  -20.546      0.000
50%           2.375       237.500   -0.027   -1.099    0.031   -0.863   -0.346 

Counting Labels: 100%|██████████| 8059/8059 [00:17<00:00, 447.95it/s]


--- Final Dataset Balance ---
Total Normal Frames (0s): 2,929,997
Total Fall Frames (1s):   549,576
Total Combined Frames:    3,479,573

New Frame-Level Ratio (Normal : Fall) -> 5.33 : 1


comparing with the original dataset


In [5]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

print("=== Starting Global Distribution Comparison ===")

def get_dataset_stats(folder_name):
    target_dir = Path(folder_name)
    all_files = list(target_dir.rglob("*.csv"))
    
    # The 9 physical sensor columns we mathematically altered
    sensor_cols = ['AccX', 'AccY', 'AccZ', 'GyrX', 'GyrY', 'GyrZ', 'EulerX', 'EulerY', 'EulerZ']
    
    df_list = []
    # Read only specific columns to keep memory usage low
    for file in tqdm(all_files, desc=f"Loading {folder_name}"):
        try:
            df = pd.read_csv(file, usecols=sensor_cols)
            df_list.append(df)
        except Exception:
            pass
            
    # Concatenate all files into one massive DataFrame
    massive_df = pd.concat(df_list, ignore_index=True)
    
    # Get describe(), transpose it for readability, and grab only the core stats
    stats = massive_df.describe().round(3).T
    return stats[['mean', 'std', 'min', 'max']]

# 1. Get stats for the Original Dataset
print("\n--- Processing Original Dataset ---")
stats_original = get_dataset_stats("./DataSet/Sample_Training")

# 2. Get stats for the Augmented Dataset
print("\n--- Processing Balanced Dataset ---")
stats_balanced = get_dataset_stats("./Balanced_DataSets/Sample_Training_Balanced")

# 3. Print the Professional Side-by-Side Comparison
print("\n" + "="*70)
print("      GLOBAL PHYSICS COMPARISON: ORIGINAL vs. BALANCED")
print("="*70)

# We will loop through the columns and print them next to each other
for col in stats_original.index:
    print(f"\n[{col}]")
    print(f"         {'ORIGINAL':<20} | {'BALANCED (AUGMENTED)':<20}")
    print(f"  Mean : {stats_original.loc[col, 'mean']:<20} | {stats_balanced.loc[col, 'mean']:<20}")
    print(f"  Std  : {stats_original.loc[col, 'std']:<20} | {stats_balanced.loc[col, 'std']:<20}")
    print(f"  Min  : {stats_original.loc[col, 'min']:<20} | {stats_balanced.loc[col, 'min']:<20}")
    print(f"  Max  : {stats_original.loc[col, 'max']:<20} | {stats_balanced.loc[col, 'max']:<20}")

print("\n" + "="*70)

=== Starting Global Distribution Comparison ===

--- Processing Original Dataset ---


Loading ./DataSet/Sample_Training: 100%|██████████| 2473/2473 [00:08<00:00, 299.82it/s]



--- Processing Balanced Dataset ---


Loading ./Balanced_DataSets/Sample_Training_Balanced: 100%|██████████| 8059/8059 [00:18<00:00, 425.68it/s] 



      GLOBAL PHYSICS COMPARISON: ORIGINAL vs. BALANCED

[AccX]
         ORIGINAL             | BALANCED (AUGMENTED)
  Mean : -0.002               | 0.0                 
  Std  : 0.241                | 0.27                
  Min  : -4.246               | -4.8                
  Max  : 4.286                | 4.807               

[AccY]
         ORIGINAL             | BALANCED (AUGMENTED)
  Mean : -0.917               | -0.903              
  Std  : 0.356                | 0.389               
  Min  : -4.325               | -4.801              
  Max  : 4.129                | 4.586               

[AccZ]
         ORIGINAL             | BALANCED (AUGMENTED)
  Mean : -0.075               | -0.069              
  Std  : 0.357                | 0.375               
  Min  : -4.307               | -4.926              
  Max  : 4.414                | 4.943               

[GyrX]
         ORIGINAL             | BALANCED (AUGMENTED)
  Mean : -5.273               | -8.199              
  Std  : 41

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

print("=== Starting Window Extraction & Labeling ===")

def extract_balanced_windows(dataset_path, window_size=50, normal_stride=25, fall_stride=5, fall_threshold=0.4):
    """
    Extracts 50-step windows and labels them.
    Uses dynamic striding to overlap falls heavily.
    """
    target_dir = Path(dataset_path)
    if not target_dir.exists():
        raise FileNotFoundError(f"CRITICAL ERROR: Folder '{dataset_path}' not found!")
        
    all_files = list(target_dir.rglob("*.csv"))
    print(f"Found {len(all_files)} CSV files in '{dataset_path}'.\n")
    
    X_list = []
    y_list = []
    
    # We are using the original 9 channels for V1
    features_cols = ['AccX', 'AccY', 'AccZ', 'GyrX', 'GyrY', 'GyrZ', 'EulerX', 'EulerY', 'EulerZ']
    
    for file in tqdm(all_files, desc="Extracting & Labeling"):
        try:
            df = pd.read_csv(file)
            total_rows = len(df)
            
            # Skip files that are too short to make even one window
            if total_rows < window_size:
                continue
                
            features = df[features_cols].values
            labels = df['FallCheck'].values
            
            start = 0
            # Slide the window, dropping the incomplete tail at the end
            while start + window_size <= total_rows:
                window_X = features[start : start + window_size]
                window_y = labels[start : start + window_size]
                
                # Labeling: If 40% of the window is a fall, label the whole window as 1
                is_fall = 1 if np.mean(window_y) >= fall_threshold else 0
                
                X_list.append(window_X)
                y_list.append(is_fall)
                
                # Dynamic Stride: Move slow over falls to extract more windows
                if is_fall == 1:
                    start += fall_stride
                else:
                    start += normal_stride
                    
        except Exception as e:
            # Silently skip unreadable files to prevent crashing
            pass
            
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

# ==========================================
# Execute the Extraction
# ==========================================
folder_to_process = "./Balanced_DataSets/Sample_Training_Balanced"

X_train_balanced, y_train_balanced = extract_balanced_windows(folder_to_process)

# ==========================================
# Verify the Results
# ==========================================
print("\n" + "="*50)
print("      FINAL WINDOW TENSOR VERIFICATION")
print("="*50)
print(f"Extracted Features Shape (X): {X_train_balanced.shape}")
print(f"Extracted Labels Shape (y):   {y_train_balanced.shape}")

num_zeros = np.sum(y_train_balanced == 0)
num_ones = np.sum(y_train_balanced == 1)

print("\n--- Final Window Balance ---")
print(f"Normal Windows (0): {num_zeros:,}")
print(f"Fall Windows (1):   {num_ones:,}")

if num_ones > 0:
    ratio = num_zeros / num_ones
    print(f"\nFinal Neural Network Ratio -> {ratio:.2f} : 1 (Normal : Fall)")

=== Starting Window Extraction & Labeling ===
Found 2473 CSV files in './Balanced_DataSets/Sample_Training_Balanced'.



Extracting & Labeling: 100%|██████████| 2473/2473 [00:27<00:00, 90.92it/s] 



      FINAL WINDOW TENSOR VERIFICATION
Extracted Features Shape (X): (58373, 50, 9)
Extracted Labels Shape (y):   (58373,)

--- Final Window Balance ---
Normal Windows (0): 41,571
Fall Windows (1):   16,802

Final Neural Network Ratio -> 2.47 : 1 (Normal : Fall)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import Dataset, DataLoader

print("=== Phase 2: Train/Validation Split & Scaling ===")

# ==========================================
# 1. Carve out a Validation Set (80/20 Split)
# ==========================================
print("Splitting the balanced windows into 80% Train and 20% Validation...")
X_train, X_val, y_train, y_val = train_test_split(
    X_train_balanced, 
    y_train_balanced, 
    test_size=0.20, 
    random_state=42, 
    stratify=y_train_balanced # Ensures the 1.65 ratio stays the same in both splits
)

print(f"\nTraining Shapes   -> X: {X_train.shape} | y: {y_train.shape}")
print(f"Validation Shapes -> X: {X_val.shape} | y: {y_val.shape}")

# ==========================================
# 2. Scaling the Data (Zero Leakage)
# ==========================================
print("\nFitting StandardScaler strictly to the Training Split...")
scaler = StandardScaler()

# Reshape, Fit on Train, Transform Train
num_train, time_steps, num_features = X_train.shape
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, num_features)).reshape(num_train, time_steps, num_features)

# Transform Validation using the Train scaler
num_val = X_val.shape[0]
X_val_scaled = scaler.transform(X_val.reshape(-1, num_features)).reshape(num_val, time_steps, num_features)

print("Scaling complete!")

# ==========================================
# 3. Wrapping in PyTorch DataLoaders
# ==========================================
class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx].T, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)

# Create the DataLoaders
batch_size = 128
train_loader = DataLoader(FallDataset(X_train_scaled, y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(FallDataset(X_val_scaled, y_val), batch_size=batch_size, shuffle=False)

print("\n=== Phase 2 Complete: Train and Val are ready. Test data remains hidden! ===")

=== Phase 2: Train/Validation Split & Scaling ===
Splitting the balanced windows into 80% Train and 20% Validation...

Training Shapes   -> X: (46698, 50, 9) | y: (46698,)
Validation Shapes -> X: (11675, 50, 9) | y: (11675,)

Fitting StandardScaler strictly to the Training Split...
Scaling complete!

=== Phase 2 Complete: Train and Val are ready. Test data remains hidden! ===


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
import numpy as np
import warnings

warnings.filterwarnings('ignore')

print("=== Defining the V1 Model (9-Channels) ===")

# 1. Check for the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")

# 2. Define the V1 Architecture 
class FallDetector1DCNN_V1(nn.Module):
    def __init__(self):
        super(FallDetector1DCNN_V1, self).__init__()
        # 9 Channels coming in (AccX, AccY, AccZ, GyrX, GyrY, GyrZ, EulerX, EulerY, EulerZ)
        self.conv1 = nn.Conv1d(in_channels=9, out_channels=64, kernel_size=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3)
        self.bn2 = nn.BatchNorm1d(128)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        self.fc1 = nn.Linear(in_features=128, out_features=64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(in_features=64, out_features=1)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.global_pool(F.relu(self.bn2(self.conv2(x))))
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)

# Initialize the model and send it to the GPU
model = FallDetector1DCNN_V1().to(device)

# 3. Calculate Dynamic pos_weight strictly from the 80% Training Split
# We use the y_train variable you created in the last step
num_zeros_train = np.sum(y_train == 0)
num_ones_train = np.sum(y_train == 1)
pos_weight_val = num_zeros_train / num_ones_train
pos_weight_tensor = torch.tensor([pos_weight_val], dtype=torch.float32).to(device)

# 4. Loss and Optimizer
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ==========================================
# 5. The Training & Validation Loop
# ==========================================
epochs = 30
print(f"\n=== Starting Training Loop ===")
print(f"Training on {len(y_train):,} windows | Validating on {len(y_val):,} windows")
print(f"Calculated pos_weight for this split: {pos_weight_val:.2f}\n")

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for val_X, val_y in val_loader:
            val_X, val_y = val_X.to(device), val_y.to(device)
            
            val_logits = model(val_X)
            v_loss = criterion(val_logits, val_y)
            val_loss += v_loss.item()
            
            val_probs = torch.sigmoid(val_logits)
            preds = (val_probs > 0.5).float()
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(val_y.cpu().numpy())
            
    avg_val_loss = val_loss / len(val_loader)
    
    # Metrics Calculation
    val_f1 = f1_score(all_targets, all_preds)
    val_precision = precision_score(all_targets, all_preds, zero_division=0)
    val_recall = recall_score(all_targets, all_preds, zero_division=0)
    
    # Print progress
    print(f"Epoch [{epoch+1:02d}/{epochs}] | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val F1: {val_f1:.4f} "
          f"(Precision: {val_precision:.4f}, Recall: {val_recall:.4f})")

print("\n=== Validation Complete ===")
print("\nValidation Confusion Matrix:")
cm = confusion_matrix(all_targets, all_preds)
print(f"True Negatives: {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")
print(f"True Positives: {cm[1][1]}")

print("\nValidation Classification Report:")
print(classification_report(all_targets, all_preds, target_names=['Normal (0)', 'Fall (1)']))

=== Defining the V1 Model (9-Channels) ===
Compute Device: cuda

=== Starting Training Loop ===
Training on 46,698 windows | Validating on 11,675 windows
Calculated pos_weight for this split: 2.47

Epoch [01/30] | Train Loss: 0.2085 | Val Loss: 0.1371 | Val F1: 0.9388 (Precision: 0.9215, Recall: 0.9569)
Epoch [02/30] | Train Loss: 0.1361 | Val Loss: 0.1139 | Val F1: 0.9508 (Precision: 0.9360, Recall: 0.9661)
Epoch [03/30] | Train Loss: 0.1182 | Val Loss: 0.1058 | Val F1: 0.9534 (Precision: 0.9338, Recall: 0.9738)
Epoch [04/30] | Train Loss: 0.1126 | Val Loss: 0.1008 | Val F1: 0.9473 (Precision: 0.9161, Recall: 0.9807)
Epoch [05/30] | Train Loss: 0.1045 | Val Loss: 0.0928 | Val F1: 0.9544 (Precision: 0.9342, Recall: 0.9756)
Epoch [06/30] | Train Loss: 0.0976 | Val Loss: 0.0998 | Val F1: 0.9518 (Precision: 0.9265, Recall: 0.9786)
Epoch [07/30] | Train Loss: 0.0959 | Val Loss: 0.0956 | Val F1: 0.9567 (Precision: 0.9315, Recall: 0.9833)
Epoch [08/30] | Train Loss: 0.0958 | Val Loss: 0.0925

In [14]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix

print("=== Phase 3: Extracting and Testing the BLIND Hold-Out Set ===")

# 1. Extract the clean test data
# Using the exact path you provided earlier
folder_test = "./DataSet/Sample_Test" 
print(f"Extracting blind test data from '{folder_test}'...")

X_test_blind, y_test_blind = extract_balanced_windows(
    folder_test, 
    window_size=50, 
    normal_stride=25, 
    fall_stride=25, # Strict stride for realistic testing!
    fall_threshold=0.4
)

# 2. Scale the test data strictly using the Train scaler
num_test_blind = X_test_blind.shape[0]
time_steps = X_test_blind.shape[1]
num_features = X_test_blind.shape[2]

X_test_blind_2d = X_test_blind.reshape(-1, num_features)
X_test_scaled_blind = scaler.transform(X_test_blind_2d).reshape(num_test_blind, time_steps, num_features)

# 3. Wrap in DataLoader
test_loader = DataLoader(FallDataset(X_test_scaled_blind, y_test_blind), batch_size=128, shuffle=False)

# 4. Run the Blind Inference
print("\nRunning inference on the strictly unseen test set...")
model.eval()
test_preds = []
test_targets = []

with torch.no_grad():
    for test_X, test_y in test_loader:
        test_X, test_y = test_X.to(device), test_y.to(device)
        
        test_logits = model(test_X)
        probs = torch.sigmoid(test_logits)
        preds = (probs > 0.5).float()
        
        test_preds.extend(preds.cpu().numpy())
        test_targets.extend(test_y.cpu().numpy())

# Final Metrics
print("\n==================================================")
print("     V1 MODEL (AUGMENTED) BLIND TEST RESULTS      ")
print("==================================================")
print(classification_report(test_targets, test_preds, target_names=['Normal (0)', 'Fall (1)']))

cm_test = confusion_matrix(test_targets, test_preds)
print("\nBlind Test Confusion Matrix:")
print(f"True Negatives: {cm_test[0][0]}")
print(f"False Positives: {cm_test[0][1]}")
print(f"False Negatives: {cm_test[1][0]}")
print(f"True Positives: {cm_test[1][1]}")

=== Phase 3: Extracting and Testing the BLIND Hold-Out Set ===
Extracting blind test data from './DataSet/Sample_Test'...
Found 586 CSV files in './DataSet/Sample_Test'.



Extracting & Labeling: 100%|██████████| 586/586 [00:06<00:00, 97.31it/s] 



Running inference on the strictly unseen test set...

     V1 MODEL (AUGMENTED) BLIND TEST RESULTS      
              precision    recall  f1-score   support

  Normal (0)       0.99      0.98      0.99      9674
    Fall (1)       0.82      0.94      0.87       967

    accuracy                           0.98     10641
   macro avg       0.91      0.96      0.93     10641
weighted avg       0.98      0.98      0.98     10641


Blind Test Confusion Matrix:
True Negatives: 9470
False Positives: 204
False Negatives: 59
True Positives: 908


now without pos_weights ( algorithmic way to handle imbalance)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import warnings

warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=== Starting Bare-Metal Training (Undersampled, No pos_weight) ===")

# 1. The Bare-Metal V1 Architecture
class FallDetector1DCNN_Bare(nn.Module):
    def __init__(self):
        super(FallDetector1DCNN_Bare, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=9, out_channels=64, kernel_size=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3)
        self.bn2 = nn.BatchNorm1d(128)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        self.fc1 = nn.Linear(in_features=128, out_features=64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(in_features=64, out_features=1)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.global_pool(F.relu(self.bn2(self.conv2(x))))
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)

model = FallDetectorConvTransformer().to(device)

# 2. NAIVE LOSS FUNCTION: Zero algorithmic help
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Assuming you already wrapped your X_train, y_train, X_val, y_val in DataLoaders
# If not, use this quick setup:
class QuickDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx].T, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)

# Uncomment these if you haven't built the loaders yet!
# train_loader = DataLoader(QuickDataset(X_train_scaled, y_train), batch_size=128, shuffle=True)
# val_loader = DataLoader(QuickDataset(X_val_scaled, y_val), batch_size=128, shuffle=False)

# 3. The Training Loop
epochs = 30
print(f"Firing up the GPU on {device}...")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch_X), batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0.0
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for val_X, val_y in val_loader:
            val_X, val_y = val_X.to(device), val_y.to(device)
            val_logits = model(val_X)
            v_loss = criterion(val_logits, val_y)
            val_loss += v_loss.item()
            
            preds = (torch.sigmoid(val_logits) > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(val_y.cpu().numpy())
            
    avg_val_loss = val_loss / len(val_loader)
    val_f1 = f1_score(all_targets, all_preds)
    val_prec = precision_score(all_targets, all_preds, zero_division=0)
    val_rec = recall_score(all_targets, all_preds, zero_division=0)
    
    if epoch == 0 or (epoch + 1) % 5 == 0 or epoch == epochs - 1:
        print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val F1: {val_f1:.4f} (Prec: {val_prec:.4f}, Rec: {val_rec:.4f})")

print("\n=== Bare-Metal Experiment Complete ===")
cm = confusion_matrix(all_targets, all_preds)
print("\nValidation Confusion Matrix:")
print(f"True Negatives: {cm[0][0]} | False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]} | True Positives: {cm[1][1]}")

=== Starting Bare-Metal Training (Undersampled, No pos_weight) ===
Firing up the GPU on cuda...
Epoch [01/30] | Train Loss: 0.1411 | Val Loss: 0.0930 | Val F1: 0.9358 (Prec: 0.9119, Rec: 0.9610)
Epoch [05/30] | Train Loss: 0.0691 | Val Loss: 0.0609 | Val F1: 0.9606 (Prec: 0.9604, Rec: 0.9607)
Epoch [10/30] | Train Loss: 0.0558 | Val Loss: 0.0517 | Val F1: 0.9646 (Prec: 0.9610, Rec: 0.9682)
Epoch [15/30] | Train Loss: 0.0496 | Val Loss: 0.0524 | Val F1: 0.9652 (Prec: 0.9567, Rec: 0.9738)
Epoch [20/30] | Train Loss: 0.0465 | Val Loss: 0.0446 | Val F1: 0.9682 (Prec: 0.9662, Rec: 0.9702)
Epoch [25/30] | Train Loss: 0.0444 | Val Loss: 0.0436 | Val F1: 0.9687 (Prec: 0.9581, Rec: 0.9795)
Epoch [30/30] | Train Loss: 0.0389 | Val Loss: 0.0420 | Val F1: 0.9730 (Prec: 0.9741, Rec: 0.9720)

=== Bare-Metal Experiment Complete ===

Validation Confusion Matrix:
True Negatives: 8227 | False Positives: 87
False Negatives: 94 | True Positives: 3267


In [12]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix

print("=== Phase 3: The Brutal Blind Test (Bare-Metal Model) ===")

# 1. Extract the UNTOUCHED, heavily imbalanced real-world test data
folder_test = "./DataSet/Sample_Test" 
print(f"Extracting brutal real-world test data from '{folder_test}'...")

# We use the standard extraction function. No undersampling here.
# Strict strides: 25 for both to simulate continuous real-world monitoring.
X_test_blind, y_test_blind = extract_balanced_windows(
    folder_test, 
    window_size=50, 
    normal_stride=25, 
    fall_stride=25, 
    fall_threshold=0.4
)

# 2. Scale the test data using the Train scaler
# (The scaler that only saw your undersampled, distorted reality)
num_test_blind = X_test_blind.shape[0]
time_steps = X_test_blind.shape[1]
num_features = X_test_blind.shape[2]

X_test_blind_2d = X_test_blind.reshape(-1, num_features)
X_test_scaled_blind = scaler.transform(X_test_blind_2d).reshape(num_test_blind, time_steps, num_features)

# 3. Wrap in DataLoader
test_loader = DataLoader(QuickDataset(X_test_scaled_blind, y_test_blind), batch_size=128, shuffle=False)

# 4. Run the Blind Inference
print("\nRunning inference...")
model.eval()
test_preds = []
test_targets = []

with torch.no_grad():
    for test_X, test_y in test_loader:
        test_X, test_y = test_X.to(device), test_y.to(device)
        
        test_logits = model(test_X)
        probs = torch.sigmoid(test_logits)
        preds = (probs > 0.5).float()
        
        test_preds.extend(preds.cpu().numpy())
        test_targets.extend(test_y.cpu().numpy())

# Final Metrics
print("\n==================================================")
print("     BARE-METAL MODEL BLIND TEST RESULTS          ")
print("==================================================")
print(classification_report(test_targets, test_preds, target_names=['Normal (0)', 'Fall (1)']))

cm_test = confusion_matrix(test_targets, test_preds)
print("\nBlind Test Confusion Matrix:")
print(f"True Negatives: {cm_test[0][0]}")
print(f"False Positives: {cm_test[0][1]}  <-- Watch this number.")
print(f"False Negatives: {cm_test[1][0]}")
print(f"True Positives: {cm_test[1][1]}")

=== Phase 3: The Brutal Blind Test (Bare-Metal Model) ===
Extracting brutal real-world test data from './DataSet/Sample_Test'...
Found 586 CSV files in './DataSet/Sample_Test'.



Extracting & Labeling: 100%|██████████| 586/586 [00:06<00:00, 84.73it/s] 



Running inference...

     BARE-METAL MODEL BLIND TEST RESULTS          
              precision    recall  f1-score   support

  Normal (0)       0.99      0.99      0.99      9674
    Fall (1)       0.87      0.91      0.89       967

    accuracy                           0.98     10641
   macro avg       0.93      0.95      0.94     10641
weighted avg       0.98      0.98      0.98     10641


Blind Test Confusion Matrix:
True Negatives: 9537
False Positives: 137  <-- Watch this number.
False Negatives: 89
True Positives: 878


# END

Down below are some experimentation work on Multi-Headed Attention , Use of Transformers.
The Structured work on those model are implemented in [using_trasnformer](using_transformer.ipynb) file. Please refer to that.

# Trying to handle imbalance with the constraint of not modifiying the minority class

# Using transformer

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

print("=== Defining the CNN-Transformer Hybrid Architecture ===")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0) # Shape: (1, max_len, d_model)

    def forward(self, x):
        # x shape: (Batch, Seq_Len, Embed_Dim)
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :].to(x.device)
        return x

class FallDetectorConvTransformer(nn.Module):
    def __init__(self, num_channels=9, cnn_filters=64, d_model=128, nhead=4, num_layers=2, dropout=0.4):
        super(FallDetectorConvTransformer, self).__init__()
        
        # -------------------------------------------
        # 1. The CNN Front-End (Feature Extraction)
        # -------------------------------------------
        # Input: (Batch, Channels=10, SeqLen=50)
        self.conv1 = nn.Conv1d(in_channels=num_channels, out_channels=cnn_filters, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(cnn_filters)
        self.pool = nn.MaxPool1d(kernel_size=2) # Reduces SeqLen from 50 to 25
        
        self.conv2 = nn.Conv1d(in_channels=cnn_filters, out_channels=d_model, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(d_model)
        
        # -------------------------------------------
        # 2. The Positional Encoding
        # -------------------------------------------
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=25)
        
        # -------------------------------------------
        # 3. The Transformer Back-End
        # -------------------------------------------
        # batch_first=True makes it expect (Batch, SeqLen, EmbedDim)
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model*2, 
            dropout=dropout, 
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        # -------------------------------------------
        # 4. The Classification Head
        # -------------------------------------------
        self.fc1 = nn.Linear(d_model, 64)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # x is (Batch, 10, 50)
        
        # --- CNN Phase ---
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)                   # Shape becomes (Batch, 64, 25)
        x = F.relu(self.bn2(self.conv2(x))) # Shape becomes (Batch, 128, 25)
        
        # --- Preparation for Transformer ---
        # CNN outputs (Batch, Channels, SeqLen). 
        # Transformer expects (Batch, SeqLen, Channels). We must permute.
        x = x.permute(0, 2, 1)             # Shape becomes (Batch, 25, 128)
        
        # Add timestamp context
        x = self.pos_encoder(x)
        
        # --- Transformer Phase ---
        x = self.transformer_encoder(x)    # Shape remains (Batch, 25, 128)
        
        # --- Aggregation ---
        # Take the average across all 25 time steps to get a single global vector
        x = x.mean(dim=1)                  # Shape becomes (Batch, 128)
        
        # --- Classification Phase ---
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)

# Quick sanity check to ensure the tensor math doesn't crash
try:
    dummy_model = FallDetectorConvTransformer()
    dummy_input = torch.randn(32, 10, 50) # Batch of 32, 10 channels, 50 steps
    dummy_output = dummy_model(dummy_input)
    print(f"Architecture successfully initialized. Output shape: {dummy_output.shape} (Expected: [32])")
except Exception as e:
    print(f"Architecture failed: {e}")

=== Defining the CNN-Transformer Hybrid Architecture ===
Architecture failed: Given groups=1, weight of size [64, 9, 3], expected input[32, 10, 50] to have 9 channels, but got 10 channels instead


In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import warnings

warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=== Starting Bare-Metal Training (Undersampled, No pos_weight) ===")

# 1. The Bare-Metal V1 Architecture
class FallDetector1DCNN_Bare(nn.Module):
    def __init__(self):
        super(FallDetector1DCNN_Bare, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=9, out_channels=64, kernel_size=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3)
        self.bn2 = nn.BatchNorm1d(128)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        self.fc1 = nn.Linear(in_features=128, out_features=64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(in_features=64, out_features=1)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.global_pool(F.relu(self.bn2(self.conv2(x))))
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)

model = FallDetectorConvTransformer().to(device)

# 2. NAIVE LOSS FUNCTION: Zero algorithmic help
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Assuming you already wrapped your X_train, y_train, X_val, y_val in DataLoaders
# If not, use this quick setup:
class QuickDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx].T, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)

# Uncomment these if you haven't built the loaders yet!
# train_loader = DataLoader(QuickDataset(X_train_scaled, y_train), batch_size=128, shuffle=True)
# val_loader = DataLoader(QuickDataset(X_val_scaled, y_val), batch_size=128, shuffle=False)

# 3. The Training Loop
epochs = 30
print(f"Firing up the GPU on {device}...")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch_X), batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0.0
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for val_X, val_y in val_loader:
            val_X, val_y = val_X.to(device), val_y.to(device)
            val_logits = model(val_X)
            v_loss = criterion(val_logits, val_y)
            val_loss += v_loss.item()
            
            preds = (torch.sigmoid(val_logits) > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(val_y.cpu().numpy())
            
    avg_val_loss = val_loss / len(val_loader)
    val_f1 = f1_score(all_targets, all_preds)
    val_prec = precision_score(all_targets, all_preds, zero_division=0)
    val_rec = recall_score(all_targets, all_preds, zero_division=0)
    
    if epoch == 0 or (epoch + 1) % 5 == 0 or epoch == epochs - 1:
        print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val F1: {val_f1:.4f} (Prec: {val_prec:.4f}, Rec: {val_rec:.4f})")

print("\n=== Bare-Metal Experiment Complete ===")
cm = confusion_matrix(all_targets, all_preds)
print("\nValidation Confusion Matrix:")
print(f"True Negatives: {cm[0][0]} | False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]} | True Positives: {cm[1][1]}")

=== Starting Bare-Metal Training (Undersampled, No pos_weight) ===
Firing up the GPU on cuda...
Epoch [01/30] | Train Loss: 0.1093 | Val Loss: 0.0687 | Val F1: 0.9553 (Prec: 0.9421, Rec: 0.9688)
Epoch [05/30] | Train Loss: 0.0531 | Val Loss: 0.0522 | Val F1: 0.9672 (Prec: 0.9564, Rec: 0.9783)
Epoch [10/30] | Train Loss: 0.0493 | Val Loss: 0.0503 | Val F1: 0.9692 (Prec: 0.9657, Rec: 0.9726)
Epoch [15/30] | Train Loss: 0.0418 | Val Loss: 0.0504 | Val F1: 0.9706 (Prec: 0.9771, Rec: 0.9643)
Epoch [20/30] | Train Loss: 0.0367 | Val Loss: 0.0472 | Val F1: 0.9734 (Prec: 0.9713, Rec: 0.9756)
Epoch [25/30] | Train Loss: 0.0334 | Val Loss: 0.0507 | Val F1: 0.9717 (Prec: 0.9673, Rec: 0.9762)
Epoch [30/30] | Train Loss: 0.0302 | Val Loss: 0.0657 | Val F1: 0.9712 (Prec: 0.9797, Rec: 0.9628)

=== Bare-Metal Experiment Complete ===

Validation Confusion Matrix:
True Negatives: 8247 | False Positives: 67
False Negatives: 125 | True Positives: 3236


In [20]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix

print("=== Phase 3: The Brutal Blind Test (Bare-Metal Model) ===")

# 1. Extract the UNTOUCHED, heavily imbalanced real-world test data
folder_test = "./DataSet/Sample_Test" 
print(f"Extracting brutal real-world test data from '{folder_test}'...")

# We use the standard extraction function. No undersampling here.
# Strict strides: 25 for both to simulate continuous real-world monitoring.
X_test_blind, y_test_blind = extract_balanced_windows(
    folder_test, 
    window_size=50, 
    normal_stride=25, 
    fall_stride=25, 
    fall_threshold=0.4
)

# 2. Scale the test data using the Train scaler
# (The scaler that only saw your undersampled, distorted reality)
num_test_blind = X_test_blind.shape[0]
time_steps = X_test_blind.shape[1]
num_features = X_test_blind.shape[2]

X_test_blind_2d = X_test_blind.reshape(-1, num_features)
X_test_scaled_blind = scaler.transform(X_test_blind_2d).reshape(num_test_blind, time_steps, num_features)

# 3. Wrap in DataLoader
test_loader = DataLoader(QuickDataset(X_test_scaled_blind, y_test_blind), batch_size=128, shuffle=False)

# 4. Run the Blind Inference
print("\nRunning inference...")
model.eval()
test_preds = []
test_targets = []

with torch.no_grad():
    for test_X, test_y in test_loader:
        test_X, test_y = test_X.to(device), test_y.to(device)
        
        test_logits = model(test_X)
        probs = torch.sigmoid(test_logits)
        preds = (probs > 0.5).float()
        
        test_preds.extend(preds.cpu().numpy())
        test_targets.extend(test_y.cpu().numpy())

# Final Metrics
print("\n==================================================")
print("     With using CNN-Transformer Hybrid Architecture BLIND TEST RESULTS          ")
print("==================================================")
print(classification_report(test_targets, test_preds, target_names=['Normal (0)', 'Fall (1)']))

cm_test = confusion_matrix(test_targets, test_preds)
print("\nBlind Test Confusion Matrix:")
print(f"True Negatives: {cm_test[0][0]}")
print(f"False Positives: {cm_test[0][1]}  <-- Watch this number.")
print(f"False Negatives: {cm_test[1][0]}")
print(f"True Positives: {cm_test[1][1]}")

=== Phase 3: The Brutal Blind Test (Bare-Metal Model) ===
Extracting brutal real-world test data from './DataSet/Sample_Test'...
Found 586 CSV files in './DataSet/Sample_Test'.



Extracting & Labeling: 100%|██████████| 586/586 [00:05<00:00, 100.80it/s]



Running inference...

     With using CNN-Transformer Hybrid Architecture BLIND TEST RESULTS          
              precision    recall  f1-score   support

  Normal (0)       0.99      0.99      0.99      9674
    Fall (1)       0.90      0.90      0.90       967

    accuracy                           0.98     10641
   macro avg       0.95      0.95      0.95     10641
weighted avg       0.98      0.98      0.98     10641


Blind Test Confusion Matrix:
True Negatives: 9578
False Positives: 96  <-- Watch this number.
False Negatives: 94
True Positives: 873


In [23]:
import numpy as np
from sklearn.metrics import precision_recall_curve, fbeta_score

def lock_safety_threshold(y_true, y_probs, target_recall=0.98):
    """
    Scans the Precision-Recall curve to find the maximum possible decision threshold
    that strictly satisfies the target Recall requirement.
    """
    y_true = np.array(y_true)
    y_probs = np.array(y_probs)
    
    # Calculate Precision and Recall across all possible thresholds
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    
    # Scikit-learn appends a final (Precision=1, Recall=0) without a threshold. 
    # We slice [:-1] to align the arrays.
    valid_indices = np.where(recalls[:-1] >= target_recall)[0]
    
    if len(valid_indices) == 0:
        print(f"CRITICAL: The model cannot achieve a Recall of {target_recall} even at a threshold of 0.0.")
        print("This means the model's highest probability for some actual falls is absolutely zero.")
        return None
        
    # We want the HIGHEST possible threshold that still maintains our safety target.
    # Because thresholds are sorted in ascending order, we take the last valid index.
    best_index = valid_indices[-1] 
    optimal_threshold = thresholds[best_index]
    
    expected_precision = precisions[best_index]
    expected_recall = recalls[best_index]
    
    # --- Simulate the Real-World Impact ---
    y_pred_simulated = (y_probs >= optimal_threshold).astype(int)
    f2 = fbeta_score(y_true, y_pred_simulated, beta=2.0)
    
    false_negatives = np.sum((y_pred_simulated == 0) & (y_true == 1))
    false_positives = np.sum((y_pred_simulated == 1) & (y_true == 0))
    
    # Output the Engineering Report
    print("\n" + "="*50)
    print(f"🚨 TARGET RECALL SECURED: {target_recall * 100}%")
    print("="*50)
    print(f"Hardcode this Threshold : {optimal_threshold:.4f}")
    print(f"Resulting Precision     : {expected_precision:.4f}")
    print(f"Resulting F2 Score      : {f2:.4f}")
    print("-" * 50)
    print("SIMULATED CONFUSION MATRIX (At new threshold):")
    print(f"Missed Falls (FN)       : {false_negatives} (This is your safety metric)")
    print(f"False Alarms (FP)       : {false_positives} (This is the cost of doing business)")
    print("="*50)
    
    return optimal_threshold

# --- Execution Example ---
# Assuming you have collected all predictions from your test loader:
# y_test_true = [0, 1, 0, 0, 1, ...]
# y_test_probs = [0.01, 0.88, 0.12, 0.05, 0.45, ...]

optimal_t = lock_safety_threshold(test_targets, probs, target_recall=0.98)

TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [24]:
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np
import math
import warnings

warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=== DEPLOYING BAYESIAN SEARCH ON CONV-TRANSFORMER ===")

# 1. Dataset Wrapper (Reused from RAM)
class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx].T, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)

# 2. The Dynamic ConvTransformer
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)

class DynamicConvTransformer(nn.Module):
    def __init__(self, d_model, dropout_rate):
        super().__init__()
        # CNN Front-End
        self.conv1 = nn.Conv1d(9, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(2) # Reduces SeqLen to 25
        
        # d_model dynamically scales the connection between CNN and Transformer
        self.conv2 = nn.Conv1d(64, d_model, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(d_model)
        
        # Transformer Back-End (Heads locked to 4)
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=25)
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, dim_feedforward=d_model*2, dropout=dropout_rate, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=2)
        
        # Classifier
        self.fc1 = nn.Linear(d_model, 64)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))
        x = x.permute(0, 2, 1) # (Batch, SeqLen, Channels)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1) 
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)

# 3. The Objective Function for Optuna
def objective(trial):
    # The Search Space
    lr = trial.suggest_float("lr", 1e-5, 5e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    dropout_rate = trial.suggest_float("dropout_rate", 0.3, 0.6)
    d_model = trial.suggest_categorical("d_model", [64, 128]) # Must be divisible by 4
    
    # Loaders
    train_loader = DataLoader(FallDataset(X_train_scaled, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(FallDataset(X_val_scaled, y_val), batch_size=batch_size, shuffle=False)
    
    # Initialize dynamic model
    model = DynamicConvTransformer(d_model=d_model, dropout_rate=dropout_rate).to(device)
    criterion = nn.BCEWithLogitsLoss() # Bare-metal: No pos_weight
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # We evaluate each trial for exactly 8 epochs.
    # If the hyperparameters are garbage, it will be obvious by epoch 8.
    EPOCHS = 8 
    
    for epoch in range(EPOCHS):
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()
            
    # Blind Validation strict metric check
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for val_X, val_y in val_loader:
            val_X, val_y = val_X.to(device), val_y.to(device)
            val_logits = model(val_X)
            preds = (torch.sigmoid(val_logits) > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(val_y.cpu().numpy())
            
    # We are forcing Optuna to maximize F1, but we will print Recall to monitor the False Negatives.
    f1 = f1_score(all_targets, all_preds)
    recall = recall_score(all_targets, all_preds, zero_division=0)
    
    # Tell Optuna if it killed too many real falls
    trial.set_user_attr("Recall", float(recall))
    
    return f1

# 4. Execute the Search
print("\n--- Commencing TPE Search (15 Trials) ---")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=15)

print("\n" + "="*50)
print("             BAYESIAN SEARCH COMPLETE")
print("="*50)
print(f"Absolute Best Validation F1: {study.best_value:.4f}")
print(f"Associated Recall (Did we miss falls?): {study.best_trial.user_attrs['Recall']:.4f}")
print("Winning Hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f" - {key}: {value}")

[I 2026-03-25 20:46:50,264] A new study created in memory with name: no-name-f477a87a-6f52-449d-a4b6-e6d5e1d27bdf


=== DEPLOYING BAYESIAN SEARCH ON CONV-TRANSFORMER ===

--- Commencing TPE Search (15 Trials) ---


[I 2026-03-25 20:48:18,548] Trial 0 finished with value: 0.9666075650118203 and parameters: {'lr': 0.0001025350969016849, 'batch_size': 64, 'dropout_rate': 0.34680559213273093, 'd_model': 64}. Best is trial 0 with value: 0.9666075650118203.
[I 2026-03-25 20:49:04,272] Trial 1 finished with value: 0.9578977932636469 and parameters: {'lr': 0.002176624112345368, 'batch_size': 128, 'dropout_rate': 0.5909729556485983, 'd_model': 64}. Best is trial 0 with value: 0.9666075650118203.
[I 2026-03-25 20:49:36,042] Trial 2 finished with value: 0.9585308056872038 and parameters: {'lr': 3.0955664602423724e-05, 'batch_size': 256, 'dropout_rate': 0.4295835055926347, 'd_model': 128}. Best is trial 0 with value: 0.9666075650118203.
[I 2026-03-25 20:50:07,775] Trial 3 finished with value: 0.9561599048892852 and parameters: {'lr': 2.3795221163877236e-05, 'batch_size': 256, 'dropout_rate': 0.5355527884179041, 'd_model': 128}. Best is trial 0 with value: 0.9666075650118203.
[I 2026-03-25 20:51:03,449] Trial


             BAYESIAN SEARCH COMPLETE
Absolute Best Validation F1: 0.9720
Associated Recall (Did we miss falls?): 0.9646
Winning Hyperparameters:
 - lr: 0.00039710847107924746
 - batch_size: 128
 - dropout_rate: 0.3195154778955838
 - d_model: 128


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import math
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"=== FORGING FINAL PRODUCTION MODEL ON {device} ===")

# 1. Hardcoding the Optuna Winners
BEST_LR = 0.00039710847107924746
BEST_BS = 128
BEST_DROP = 0.3195154778955838
BEST_D_MODEL = 128

print(f"Locked Parameters -> LR: {BEST_LR:.6f} | Drop: {BEST_DROP:.4f} | D_Model: {BEST_D_MODEL}")

# 2. The Final Architecture
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)

class ProductionConvTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(9, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(2) 
        self.conv2 = nn.Conv1d(64, BEST_D_MODEL, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(BEST_D_MODEL)
        
        self.pos_encoder = PositionalEncoding(d_model=BEST_D_MODEL, max_len=25)
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=BEST_D_MODEL, nhead=4, dim_feedforward=BEST_D_MODEL*2, dropout=BEST_DROP, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=2)
        
        self.fc1 = nn.Linear(BEST_D_MODEL, 64)
        self.dropout = nn.Dropout(BEST_DROP)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))
        x = x.permute(0, 2, 1) 
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1) 
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)

# Initialize
final_model = ProductionConvTransformer().to(device)
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(final_model.parameters(), lr=BEST_LR)

# 3. Training with Checkpointing (Saving the absolute best state)
EPOCHS = 30
best_val_f1 = 0.0
best_model_weights = None

history_train_loss = []
history_val_loss = []
history_val_recall = []
history_val_f1 = []
print("\n--- Commencing Production Training ---")
for epoch in range(EPOCHS):
    final_model.train()
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        loss = criterion(final_model(batch_X), batch_y)
        loss.backward()
        optimizer.step()
        
    final_model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for val_X, val_y in val_loader:
            val_X, val_y = val_X.to(device), val_y.to(device)
            val_logits = final_model(val_X)
            preds = (torch.sigmoid(val_logits) > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(val_y.cpu().numpy())
            
    current_f1 = f1_score(all_targets, all_preds)
    current_rec = recall_score(all_targets, all_preds, zero_division=0)
    
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Val F1: {current_f1:.4f} | Val Recall: {current_rec:.4f}")
    history_train_loss.append(train_loss / len(train_loader))
    history_val_loss.append(val_loss / len(val_loader))
    history_val_f1.append(current_f1)
    history_val_recall.append(current_rec)
    # Save the best weights
    if current_f1 > best_val_f1:
        best_val_f1 = current_f1
        best_model_weights = copy.deepcopy(final_model.state_dict())

# Load the best weights back into the model for the blind test
final_model.load_state_dict(best_model_weights)
print(f"\n[Locked in best model weights with Val F1: {best_val_f1:.4f}]")



=== FORGING FINAL PRODUCTION MODEL ON cuda ===
Locked Parameters -> LR: 0.000397 | Drop: 0.3195 | D_Model: 128

--- Commencing Production Training ---
Epoch [01/30] | Val F1: 0.9571 | Val Recall: 0.9598
Epoch [02/30] | Val F1: 0.9630 | Val Recall: 0.9729
Epoch [03/30] | Val F1: 0.9647 | Val Recall: 0.9586
Epoch [04/30] | Val F1: 0.9669 | Val Recall: 0.9747
Epoch [05/30] | Val F1: 0.9709 | Val Recall: 0.9771
Epoch [06/30] | Val F1: 0.9691 | Val Recall: 0.9646
Epoch [07/30] | Val F1: 0.9697 | Val Recall: 0.9816
Epoch [08/30] | Val F1: 0.9731 | Val Recall: 0.9798
Epoch [09/30] | Val F1: 0.9637 | Val Recall: 0.9473
Epoch [10/30] | Val F1: 0.9725 | Val Recall: 0.9685
Epoch [11/30] | Val F1: 0.9727 | Val Recall: 0.9705
Epoch [12/30] | Val F1: 0.9730 | Val Recall: 0.9768
Epoch [13/30] | Val F1: 0.9758 | Val Recall: 0.9768
Epoch [14/30] | Val F1: 0.9664 | Val Recall: 0.9896
Epoch [15/30] | Val F1: 0.9717 | Val Recall: 0.9851
Epoch [16/30] | Val F1: 0.9788 | Val Recall: 0.9810
Epoch [17/30] | V

Processing Blind Test: 100%|██████████| 586/586 [00:08<00:00, 69.57it/s]


ValueError: X has 10 features, but StandardScaler is expecting 9 features as input.

In [28]:
# ==========================================
# 4. THE ULTIMATE BLIND TEST (9-CHANNEL FIX)
# ==========================================
print("\n" + "="*50)
print("     EXTRACTING REAL-WORLD BLIND TEST DATA")
print("="*50)

folder_test = Path("./DataSet/Sample_Test")
test_files = list(folder_test.rglob("*.csv"))

X_test_list, y_test_list = [], []
for file in tqdm(test_files, desc="Processing Blind Test"):
    try:
        df = pd.read_csv(file)
        if len(df) < 50: continue
        
        # WE REMOVED SMV HERE TO MATCH YOUR 9-CHANNEL SCALER
        features = df[['AccX', 'AccY', 'AccZ', 'GyrX', 'GyrY', 'GyrZ', 'EulerX', 'EulerY', 'EulerZ']].values
        labels = df['FallCheck'].values
        
        # Strict continuous extraction for the blind test
        start = 0
        while start + 50 <= len(df):
            X_test_list.append(features[start : start + 50])
            y_test_list.append(1 if np.mean(labels[start : start + 50]) >= 0.4 else 0)
            start += 25 # Strict 25-frame stride to simulate reality
    except Exception:
        pass

X_test_raw = np.array(X_test_list, dtype=np.float32)
y_test_blind = np.array(y_test_list, dtype=np.float32)

# Scale using the TRAIN scaler to prevent leakage (Now matching 9 channels!)
n_test, t_steps, n_feats = X_test_raw.shape
X_test_scaled = scaler.transform(X_test_raw.reshape(-1, n_feats)).reshape(n_test, t_steps, n_feats)

class QuickDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx].T, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)

blind_loader = DataLoader(QuickDataset(X_test_scaled, y_test_blind), batch_size=BEST_BS, shuffle=False)

# 5. Final Inference
print("\nRunning Final Inference on Blind Test Set...")
final_model.eval()
test_preds, test_targets = [], []

with torch.no_grad():
    for test_X, test_y in blind_loader:
        test_X, test_y = test_X.to(device), test_y.to(device)
        logits = final_model(test_X)
        preds = (torch.sigmoid(logits) > 0.5).float()
        test_preds.extend(preds.cpu().numpy())
        test_targets.extend(test_y.cpu().numpy())

print("\n==================================================")
print("     FINAL PRODUCTION MODEL: BLIND TEST RESULTS   ")
print("==================================================")
print(classification_report(test_targets, test_preds, target_names=['Normal (0)', 'Fall (1)']))

cm_test = confusion_matrix(test_targets, test_preds)
print("\nBlind Test Confusion Matrix:")
print(f"True Negatives: {cm_test[0][0]}")
print(f"False Positives: {cm_test[0][1]}")
print(f"False Negatives: {cm_test[1][0]}")
print(f"True Positives: {cm_test[1][1]}")


     EXTRACTING REAL-WORLD BLIND TEST DATA


Processing Blind Test: 100%|██████████| 586/586 [00:09<00:00, 62.67it/s] 



Running Final Inference on Blind Test Set...

     FINAL PRODUCTION MODEL: BLIND TEST RESULTS   
              precision    recall  f1-score   support

  Normal (0)       0.99      0.99      0.99      9674
    Fall (1)       0.88      0.93      0.91       967

    accuracy                           0.98     10641
   macro avg       0.94      0.96      0.95     10641
weighted avg       0.98      0.98      0.98     10641


Blind Test Confusion Matrix:
True Negatives: 9556
False Positives: 118
False Negatives: 68
True Positives: 899


In [ ]:
import matplotlib.pyplot as plt

print("=== GENERATING TELEMETRY DIAGNOSTICS ===")

# Create the X-axis (Epochs 1 to 30)
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(14, 5))

# --- PLOT 1: The Loss Curve (Checking for Overfitting) ---
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history_train_loss, label='Train Loss', color='#1f77b4', linewidth=2, marker='o', markersize=4)
plt.plot(epochs_range, history_val_loss, label='Validation Loss', color='#d62728', linewidth=2, marker='x', markersize=4)
plt.title('Neural Network Loss Over Time')
plt.xlabel('Epochs')
plt.ylabel('BCE Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# --- PLOT 2: The Safety Metrics (Recall & F1) ---
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history_val_recall, label='Validation Recall (Did we miss falls?)', color='#2ca02c', linewidth=2, marker='s', markersize=4)
plt.plot(epochs_range, history_val_f1, label='Validation F1-Score', color='#9467bd', linewidth=2, linestyle='--')
plt.title('Real-World Safety Metrics')
plt.xlabel('Epochs')
plt.ylabel('Score')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()